In [5]:
# ================================
# TASK 1: Data Preparation
# ================================

import pandas as pd

# Show full output (no truncation)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("=== TASK 1: Data Preparation ===\n")

# Create DataFrames
students = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None,
              'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
})

enrollments = pd.DataFrame({
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01',
                        '2024-02-10', '2024-02-15', '2024-03-01']
})

scores = pd.DataFrame({
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
})

# Display original students
print("Original Students DataFrame:")
print(students, "\n")

# Null analysis
print("Null Value Analysis:")
null_counts = students.isnull().sum()
null_percent = (null_counts / len(students)) * 100

for col in students.columns:
    print(f"Column: {col}, Nulls: {null_counts[col]} ({null_percent[col]:.2f}%)")

# Fix missing values
students['city'] = students['city'].fillna('Unknown')

# Drop missing names (safe copy)
students_clean = students.dropna(subset=['name']).copy()

print("\nCleaned Students DataFrame (6 rows):")
print(students_clean, "\n")


=== TASK 1: Data Preparation ===

Original Students DataFrame:
   student_id   name              email       city
0         101  Alice    alice@email.com     Mumbai
1         102    Bob      bob@email.com      Delhi
2         103   None  charlie@email.com  Bangalore
3         104  David               None     Mumbai
4         105   Emma     emma@email.com       None
5         106  Frank    frank@email.com    Chennai
6         107  Grace    grace@email.com      Delhi 

Null Value Analysis:
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)

Cleaned Students DataFrame (6 rows):
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
3         104  David             None   Mumbai
4         105   Emma   emma@email.com  Unknown
5         106  Frank  frank@email.com  Chennai
6         107  Grace  grace@email.com    Delhi 



In [6]:


# ================================
# TASK 2: Join Operations
# ================================

print("=== TASK 2: Join Operations ===\n")

# INNER JOIN
inner_df = pd.merge(students_clean, enrollments, on='student_id', how='inner')

print("Inner Join Result (3 rows):")
print(inner_df, "\n")

print("Students in result:", inner_df['student_id'].tolist())
print("Excluded students:", [104, 106, 107], "- Not in enrollments table\n")


# LEFT JOIN
left_df = pd.merge(students_clean, enrollments, on='student_id', how='left')

print("Left Join Result (6 rows):")
print(left_df, "\n")

null_courses = left_df[left_df['course_name'].isnull()]['student_id'].tolist()
print("Students with null course_name:", null_courses, "\n")


# RIGHT JOIN
right_df = pd.merge(students_clean, enrollments, on='student_id', how='right')

print("Right Join Result (6 rows):")
print(right_df, "\n")

missing_names = right_df[right_df['name'].isnull()]['student_id'].tolist()
print("Student IDs without names:", missing_names, "\n")


# OUTER JOIN
outer_df = pd.merge(students_clean, enrollments, on='student_id', how='outer', indicator=True)

print("Outer Join Result (9 rows):")
print(outer_df, "\n")

print("Rows with missing name OR course_name:")
print(outer_df[(outer_df['name'].isnull()) | (outer_df['course_name'].isnull())], "\n")

print("Merge Indicator Distribution:")
print(outer_df['_merge'].value_counts(), "\n")



=== TASK 2: Join Operations ===

Inner Join Result (3 rows):
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         105   Emma   emma@email.com  Unknown  Machine Learning   

  enrollment_date  
0      2024-01-15  
1      2024-01-20  
2      2024-02-10   

Students in result: [101, 102, 105]
Excluded students: [104, 106, 107] - Not in enrollments table

Left Join Result (6 rows):
   student_id   name            email     city       course_name  \
0         101  Alice  alice@email.com   Mumbai            Python   
1         102    Bob    bob@email.com    Delhi      Data Science   
2         104  David             None   Mumbai               NaN   
3         105   Emma   emma@email.com  Unknown  Machine Learning   
4         106  Frank  frank@email.com  Chennai               NaN   
5         107  Grace  grace@email.com    Delhi    

In [7]:

# ================================
# TASK 3: Lookup and Automation
# ================================

print("=== TASK 3: Lookup and Automation ===\n")

# Lookup using map
score_map = scores.set_index('student_id')['exam_score'].to_dict()
students_clean.loc[:, 'exam_score'] = students_clean['student_id'].map(score_map)

print("Lookup Operation Result:")
print(students_clean[['student_id', 'name', 'exam_score']], "\n")


# Merge method
merge_scores = pd.merge(
    students_clean.drop(columns=['exam_score'], errors='ignore'),
    scores,
    on='student_id',
    how='left'
)

print("Merge Method Result:")
print(merge_scores[['student_id', 'name', 'exam_score']], "\n")


# Performance explanation
print("Why map() is more efficient than merge():")
print("- map() uses dictionary lookup (O(1))")
print("- Faster for single column mapping")
print("- Uses less memory than merge()\n")


# ================================
# Automation Function
# ================================

def auto_merge(df1, df2, join_type, key_column):
    result = pd.merge(df1, df2, on=key_column, how=join_type)
    return {
        'result_df': result,
        'row_count': len(result),
        'join_type': join_type
    }

# Testing
test_inner = auto_merge(students_clean, enrollments, 'inner', 'student_id')
test_left = auto_merge(students_clean, enrollments, 'left', 'student_id')

print("Automation Function Test:")

print("Join Type:", test_inner['join_type'])
print("Rows in Result:", test_inner['row_count'])
print("Result Preview:")
print(test_inner['result_df'][['student_id', 'name', 'course_name']], "\n")

print("Join Type:", test_left['join_type'])
print("Rows in Result:", test_left['row_count'])
print("Result Preview:")
print(test_left['result_df'][['student_id', 'name', 'course_name']])
print("...")

=== TASK 3: Lookup and Automation ===

Lookup Operation Result:
   student_id   name  exam_score
0         101  Alice        85.0
1         102    Bob        92.0
3         104  David        78.0
4         105   Emma        88.0
5         106  Frank        95.0
6         107  Grace         NaN 

Merge Method Result:
   student_id   name  exam_score
0         101  Alice        85.0
1         102    Bob        92.0
2         104  David        78.0
3         105   Emma        88.0
4         106  Frank        95.0
5         107  Grace         NaN 

Why map() is more efficient than merge():
- map() uses dictionary lookup (O(1))
- Faster for single column mapping
- Uses less memory than merge()

Automation Function Test:
Join Type: inner
Rows in Result: 3
Result Preview:
   student_id   name       course_name
0         101  Alice            Python
1         102    Bob      Data Science
2         105   Emma  Machine Learning 

Join Type: left
Rows in Result: 6
Result Preview:
   student_id   